### Persistent Memory Notebook — Conversations + Semantic Facts with SQLite

**What you'll learn:**
- Why in-memory checkpointing is not enough
- How to persist conversation history to **SQLite** so it survives restarts
- How to build a **semantic memory store** that saves and recalls facts
- Combining both into a single agent

---
> **Prerequisite:** Complete Notebook 2 — Memory first!
---

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Install the separate checkpoint-sqlite package (required since LangGraph v0.2)
!pip install langgraph-checkpoint-sqlite -q

---
## The Problem with MemorySaver

In Notebook 2 we used `MemorySaver` — it stores checkpoints **in RAM**.

```
Session 1:  User: "My name is Rahul"  →  MemorySaver stores it ✅
            User: "What's my name?"   →  "Rahul" ✅

** Restart the kernel / script **

Session 2:  User: "What's my name?"   →  "I don't know" ❌  (RAM wiped!)
```

**Solution:** Use `SqliteSaver` — checkpoints persist to a `.db` file on disk.

Even after a kernel restart, the agent picks up right where it left off.

---
## Part 1 — SQLite Conversation Memory (Persistent Checkpointing)

In [ ]:
from typing import Annotated, Sequence, TypedDict
import operator
import sqlite3
import json
import requests
from datetime import datetime

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.sqlite import SqliteSaver

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# ─────────────────────────────────────────────────────
# Utility Tools (defined inline — no external imports)
# ─────────────────────────────────────────────────────

@tool
def get_weather(location: str) -> str:
    """Get current weather for a location.
    
    Args:
        location: City name (e.g., "New York", "London", "Mumbai")
    Returns:
        Current weather information including temperature and conditions.
    """
    url = f"https://wttr.in/{location}?format=j1"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()
    current = data["current_condition"][0]
    temp_c = current["temp_C"]
    feels_like = current["FeelsLikeC"]
    humidity = current["humidity"]
    description = current["weatherDesc"][0]["value"]
    return f"Weather in {location}: {description}, Temp: {temp_c}°C, Feels like: {feels_like}°C, Humidity: {humidity}%"


@tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression.
    
    Args:
        expression: Math expression like "2 + 2" or "15 * 7"
    """
    try:
        result = eval(expression)
        print(f"[TOOL] calculate ('{expression}') -> '{result}'")
        return str(result)
    except Exception as e:
        return f"Error: {e}"


print("✅ All imports and tools ready!")

In [ ]:
# ─────────────────────────────────────────────────────
# Build an agent with SQLite-backed persistent memory
# ─────────────────────────────────────────────────────

DB_PATH = "agent_memory.db"

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

SYSTEM_PROMPT = SystemMessage(content="""You are a helpful AI assistant with access to tools.
You remember everything the user tells you across conversations.
Always be friendly and refer to past context when relevant.""")

all_tools = [get_weather, calculate]

def agent_node(state: AgentState):
    llm_with_tools = llm.bind_tools(all_tools)
    messages = [SYSTEM_PROMPT] + list(state["messages"])
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

# Build the graph
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(all_tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

# Compile with SQLite checkpointer (persistent!)
# NOTE: Use sqlite3.connect() directly — from_conn_string() returns a context manager
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(conn)
checkpointer.setup()
app = builder.compile(checkpointer=checkpointer)

print(f"✅ Agent compiled with SQLite persistence → {DB_PATH}")

In [ ]:
# ── Helper function for chatting ──
def chat(user_input: str, thread_id: str = "default") -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config
    )
    return result["messages"][-1].content

# ── Test: Multi-turn conversation ──
print("Turn 1:", chat("My name is Rahul and I'm an ML engineer at EY.", thread_id="rahul_session"))
print()
print("Turn 2:", chat("I live in Mumbai and love cricket.", thread_id="rahul_session"))
print()
print("Turn 3:", chat("What do you know about me so far?", thread_id="rahul_session"))

---
### Inspecting the SQLite Database

The conversation checkpoints are now stored on disk. Let's peek inside:

In [ ]:
# ── Inspect the SQLite database ──
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Show all tables created by SqliteSaver
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("📦 Tables in agent_memory.db:")
for t in tables:
    print(f"   • {t[0]}")

# Count checkpoints
cursor.execute("SELECT COUNT(*) FROM checkpoints;")
count = cursor.fetchone()[0]
print(f"\n📊 Total checkpoints stored: {count}")

conn.close()
print(f"\n💾 Database file size: {os.path.getsize(DB_PATH) / 1024:.1f} KB")

---
### Persistence Test

The key advantage over `MemorySaver`: if you **restart your kernel** and re-run from the top,
then call `chat("What's my name?", thread_id="rahul_session")`, the agent will still remember!

This is because checkpoints live in `agent_memory.db`, not in RAM.

In [ ]:
# ── Persistence proof: load state from disk ──
config = {"configurable": {"thread_id": "rahul_session"}}
state = app.get_state(config)

print(f"📜 Messages recovered from SQLite ({len(state.values['messages'])} total):\n")
for i, msg in enumerate(state.values["messages"]):
    role = type(msg).__name__
    print(f"  [{i}] {role}: {msg.content[:90]}")

---
## Part 2 — Semantic Memory Store (Facts & Preferences)

Conversation checkpointing replays the full message history — great for continuity,
but it has limits:

| Conversation Memory | Semantic Memory |
|---|---|
| Stores raw messages | Stores extracted **facts** |
| Grows linearly per turn | Compact — one fact per row |
| Tied to a `thread_id` | Shared across **all** threads |
| "What did we talk about?" | "What do I **know** about this user?" |

**Semantic memory** lets the agent remember facts like "Rahul likes cricket"
even in a brand new conversation thread.

We'll build a simple SQLite-backed fact store and wire it into the agent as tools.

In [ ]:
# ─────────────────────────────────────────────────────
# Semantic Memory Store — SQLite-backed fact storage
# ─────────────────────────────────────────────────────

SEMANTIC_DB = "semantic_memory.db"

def init_semantic_db():
    """Create the facts table if it doesn't exist."""
    conn = sqlite3.connect(SEMANTIC_DB)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS facts (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id TEXT NOT NULL,
            category TEXT NOT NULL,
            fact TEXT NOT NULL,
            timestamp TEXT NOT NULL,
            UNIQUE(user_id, fact)
        )
    """)
    conn.commit()
    conn.close()

init_semantic_db()
print(f"✅ Semantic memory database ready → {SEMANTIC_DB}")

In [ ]:
# ─────────────────────────────────────────────────────
# Memory Tools — the agent uses these to save & recall
# ─────────────────────────────────────────────────────

@tool
def save_memory(user_id: str, category: str, fact: str) -> str:
    """Save a fact or preference about a user to long-term memory.
    
    Use this tool whenever the user shares personal information, preferences,
    or important details worth remembering across conversations.
    
    Args:
        user_id: Identifier for the user (e.g., "rahul", "priya")
        category: Category of the fact — one of: personal, preference, work, location, hobby, goal
        fact: The fact to remember (e.g., "Works as an ML engineer at EY")
    """
    conn = sqlite3.connect(SEMANTIC_DB)
    try:
        conn.execute(
            "INSERT OR IGNORE INTO facts (user_id, category, fact, timestamp) VALUES (?, ?, ?, ?)",
            (user_id.lower(), category.lower(), fact, datetime.now().isoformat())
        )
        conn.commit()
        print(f"[MEMORY] Saved: {category} → {fact}")
        return f"Remembered: [{category}] {fact}"
    finally:
        conn.close()


@tool
def recall_memories(user_id: str, category: str = "") -> str:
    """Recall stored facts about a user from long-term memory.
    
    Use this tool to retrieve what you know about a user before answering
    questions about them, or at the start of a new conversation.
    
    Args:
        user_id: Identifier for the user (e.g., "rahul", "priya")
        category: Optional — filter by category (personal, preference, work, etc.). Leave empty for all facts.
    """
    conn = sqlite3.connect(SEMANTIC_DB)
    try:
        if category:
            rows = conn.execute(
                "SELECT category, fact, timestamp FROM facts WHERE user_id = ? AND category = ? ORDER BY timestamp",
                (user_id.lower(), category.lower())
            ).fetchall()
        else:
            rows = conn.execute(
                "SELECT category, fact, timestamp FROM facts WHERE user_id = ? ORDER BY category, timestamp",
                (user_id.lower(),)
            ).fetchall()
        
        if not rows:
            return f"No memories found for user '{user_id}'" + (f" in category '{category}'" if category else "")
        
        memories = []
        for cat, fact, ts in rows:
            memories.append(f"  [{cat}] {fact}  (saved: {ts[:10]})")
        
        result = f"Memories for '{user_id}' ({len(rows)} facts):\n" + "\n".join(memories)
        print(f"[MEMORY] Recalled {len(rows)} facts for {user_id}")
        return result
    finally:
        conn.close()


@tool  
def forget_memory(user_id: str, fact: str) -> str:
    """Delete a specific fact from a user's long-term memory.
    
    Use when the user asks you to forget something or when information is outdated.
    
    Args:
        user_id: Identifier for the user
        fact: The exact fact text to remove
    """
    conn = sqlite3.connect(SEMANTIC_DB)
    try:
        cursor = conn.execute(
            "DELETE FROM facts WHERE user_id = ? AND fact = ?",
            (user_id.lower(), fact)
        )
        conn.commit()
        if cursor.rowcount > 0:
            print(f"[MEMORY] Deleted: {fact}")
            return f"Forgotten: {fact}"
        else:
            return f"Fact not found: {fact}"
    finally:
        conn.close()


print("✅ Memory tools defined: save_memory, recall_memories, forget_memory")

---
### Quick Test: Saving & Recalling Facts Directly

In [ ]:
# ── Test the memory tools directly (without the agent) ──

# Save some facts
save_memory.invoke({"user_id": "rahul", "category": "personal", "fact": "Name is Rahul"})
save_memory.invoke({"user_id": "rahul", "category": "work", "fact": "Works as an ML engineer at EY"})
save_memory.invoke({"user_id": "rahul", "category": "location", "fact": "Lives in Mumbai"})
save_memory.invoke({"user_id": "rahul", "category": "hobby", "fact": "Loves playing cricket"})

# Recall everything
print("\n" + recall_memories.invoke({"user_id": "rahul"}))

---
## Part 3 — Combined Agent: Conversation + Semantic Memory

Now we bring it all together — one agent that has:
1. **SQLite checkpointing** for conversation continuity
2. **Semantic memory tools** for saving/recalling facts
3. **Utility tools** (weather, calculator) defined above

In [ ]:
# ─────────────────────────────────────────────────────
# Full Agent: Conversation Memory + Semantic Memory
# ─────────────────────────────────────────────────────

FULL_SYSTEM_PROMPT = SystemMessage(content="""You are a helpful AI assistant with persistent memory.

MEMORY INSTRUCTIONS:
- When the user shares personal info, preferences, or facts → use save_memory to store them.
  Use user_id based on context (ask if unclear). Categories: personal, preference, work, location, hobby, goal.
- When the user asks "what do you know about me" or starts a new conversation → use recall_memories first.
- When the user says "forget X" → use forget_memory to delete it.

You also have access to weather and calculator tools. Always be friendly and proactive about remembering things.""")

# All tools: memory + utility
all_tools_full = [save_memory, recall_memories, forget_memory, get_weather, calculate]

def full_agent_node(state: AgentState):
    llm_with_tools = llm.bind_tools(all_tools_full)
    messages = [FULL_SYSTEM_PROMPT] + list(state["messages"])
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def full_should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

# Build the full graph
full_builder = StateGraph(AgentState)
full_builder.add_node("agent", full_agent_node)
full_builder.add_node("tools", ToolNode(all_tools_full))
full_builder.add_edge(START, "agent")
full_builder.add_conditional_edges("agent", full_should_continue, {"tools": "tools", END: END})
full_builder.add_edge("tools", "agent")

# Compile with SQLite persistence
full_conn = sqlite3.connect("full_agent_memory.db", check_same_thread=False)
full_checkpointer = SqliteSaver(full_conn)
full_checkpointer.setup()
full_app = full_builder.compile(checkpointer=full_checkpointer)

print("✅ Full agent compiled with conversation + semantic memory!")

In [ ]:
# ── Helper for the full agent ──
def full_chat(user_input: str, thread_id: str = "default") -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = full_app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config
    )
    return result["messages"][-1].content

---
### Demo: Session 1 — Meeting the User

The agent should automatically save facts as the user shares them.

In [ ]:
# ── Session 1: User introduces themselves ──
SESSION_1 = "session_intro"

turns_s1 = [
    "Hi! I'm Rahul, I work as an ML engineer at EY in Mumbai.",
    "I love playing cricket on weekends and my goal is to learn LangGraph deeply.",
    "My favorite cuisine is South Indian food.",
]

for i, turn in enumerate(turns_s1, 1):
    response = full_chat(turn, thread_id=SESSION_1)
    print(f"\n{'─'*55}")
    print(f"[Turn {i}] 👤 {turn}")
    print(f"         🤖 {response[:200]}")

---
### Demo: Session 2 — Brand New Thread, But Memories Persist!

This is the magic of semantic memory. Even in a **completely new thread**, the agent
can recall stored facts by calling `recall_memories`.

In [ ]:
# ── Session 2: Brand new thread — can the agent recall? ──
SESSION_2 = "session_new_day"

response = full_chat(
    "Hey, I'm Rahul. What do you remember about me?",
    thread_id=SESSION_2
)
print("🤖", response)

In [ ]:
# ── Use tools + memory in the same conversation ──
response = full_chat(
    "What's the weather like in Mumbai right now? Also, calculate how many weekends are in 3 months (assume 4 per month).",
    thread_id=SESSION_2
)
print("🤖", response)

---
### Demo: Forget a Memory

In [ ]:
# ── Ask the agent to forget something ──
response = full_chat(
    "Please forget that I like South Indian food.",
    thread_id=SESSION_2
)
print("🤖", response)

# Verify it's gone
print("\n── Remaining memories ──")
print(recall_memories.invoke({"user_id": "rahul"}))

---
## Inspecting the Semantic Memory Database

In [ ]:
# ── View all facts in the semantic memory database ──
conn = sqlite3.connect(SEMANTIC_DB)
cursor = conn.execute("SELECT user_id, category, fact, timestamp FROM facts ORDER BY user_id, category")
rows = cursor.fetchall()

print(f"📦 Semantic Memory Store — {len(rows)} facts total\n")
print(f"{'User':<12} {'Category':<14} {'Fact':<45} {'Saved':<12}")
print("─" * 85)
for user_id, category, fact, ts in rows:
    print(f"{user_id:<12} {category:<14} {fact:<45} {ts[:10]}")

conn.close()

---
## Architecture Summary

```
┌─────────────────────────────────────────────────────┐
│                   Full Agent                        │
│                                                     │
│  ┌──────────┐    ┌──────────┐    ┌──────────────┐   │
│  │  agent   │───▶│ tools    │───▶│  agent       │   │
│  │  node    │◀───│ node     │    │  (continue)  │   │
│  └──────────┘    └──────────┘    └──────────────┘   │
│       │               │                             │
│       │          ┌────┴────────────────┐            │
│       │          │  save_memory        │            │
│       │          │  recall_memories    │            │
│       │          │  forget_memory      │            │
│       │          │  get_weather        │            │
│       │          │  calculate          │            │
│       │          └─────────────────────┘            │
│       │                                             │
│  ┌────▼─────────────────────────────────────────┐   │
│  │         SQLite Checkpointer                  │   │
│  │   (full_agent_memory.db — conversations)     │   │
│  └──────────────────────────────────────────────┘   │
│                                                     │
│  ┌──────────────────────────────────────────────┐   │
│  │         SQLite Fact Store                    │   │
│  │   (semantic_memory.db — facts & prefs)       │   │
│  └──────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────┘
```

**Two layers of persistence:**
- **Conversation Memory** (checkpointer) → same `thread_id` = continues conversation
- **Semantic Memory** (fact store) → shared across ALL threads, survives restarts

---
## Utility: Reset All Memory

Run this cell only if you want to wipe both databases and start fresh.

In [ ]:
# ⚠️ DANGER ZONE — Uncomment and run to wipe all memory

# import os
# for db in ["agent_memory.db", "semantic_memory.db", "full_agent_memory.db"]:
#     if os.path.exists(db):
#         os.remove(db)
#         print(f"🗑️  Deleted {db}")
# print("All memory wiped! Re-run the notebook from the top.")

---
## Exercises

1. **Multi-user test** — Have two users (e.g., "rahul" and "priya") save facts in separate threads, then verify each user's memories are isolated
2. **Persistence test** — Restart the kernel, re-run from the top, and confirm the agent still remembers facts from `semantic_memory.db`
3. **Category filter** — Use `recall_memories` with a specific category (e.g., "hobby") and verify it only returns matching facts
4. **Extend it** — Add a `search_memories` tool that does fuzzy/partial text matching using SQL `LIKE`

---
**Databases created by this notebook:**
- `agent_memory.db` — Part 1 conversation checkpoints
- `semantic_memory.db` — Part 2 semantic fact store
- `full_agent_memory.db` — Part 3 combined agent checkpoints